In [1]:
import cv2
import mediapipe as mp
import numpy as np

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54)  # vibrant green
MAX_HANDS = 2
CAMERA = 0
FPS = 30
REC_TEST = False
frame_idx = 0

In [2]:
def create_hand_detector(model_path: str = "hand_landmarker.task", num_hands: int = 1):
    """
    MediaPipe HandLandmarker 생성.
    """
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=num_hands,
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
    )
    detector = vision.HandLandmarker.create_from_options(options)
    return detector

def get_drawing_utils():
    """
    랜드마크 시각화를 위한 연결 정보 반환.
    """
    return vision.HandLandmarksConnections.HAND_CONNECTIONS

def capture_frame(cap: cv2.VideoCapture):
    """
    웹캠에서 프레임 1장을 읽어 반환.
    실패 시 (False, None) 반환.
    """
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    if not ret:
        return False, None
    return True, frame

def convert_bgr_to_mp_image(bgr_frame: np.ndarray):
    """
    OpenCV BGR 이미지를 RGB 및 MediaPipe Image로 변환.
    """
    rgb_frame = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    return rgb_frame, mp_image

def detect_hand_landmarks(detector, bgr_frame: np.ndarray):
    """
    BGR 프레임을 받아 MediaPipe HandLandmarker로 손 랜드마크 검출.
    반환:
      - rgb_frame
      - detection_result
    """
    global frame_idx
    rgb_frame, mp_image = convert_bgr_to_mp_image(bgr_frame)
    timestamp_ms = int(frame_idx * 1000 / FPS)
    # detection_result = detector.detect(mp_image)
    detection_result = detector.detect_for_video(mp_image, timestamp_ms)
    frame_idx += 1


    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    for idx in range(len(hand_landmarks_list)):
        hand_label = handedness_list[idx][0].category_name

        if hand_label == "Right":
            handedness_list[idx][0].category_name = "Left"
        elif hand_label == "Left":
            handedness_list[idx][0].category_name = "Right"

    return rgb_frame, detection_result

def draw_landmarks_on_image(rgb_image, detection_result, hand_connections):
    hand_landmarks_list = detection_result.hand_landmarks
    handedness_list = detection_result.handedness
    annotated_image = np.copy(rgb_image)
    image_h, image_w, _ = annotated_image.shape

    for idx, hand_landmarks in enumerate(hand_landmarks_list):
        # 관절 연결선 그리기
        for connection in hand_connections:
            start_idx, end_idx = connection.start, connection.end
            start_lm = hand_landmarks[start_idx]
            end_lm = hand_landmarks[end_idx]
            start_point = (int(start_lm.x * image_w), int(start_lm.y * image_h))
            end_point = (int(end_lm.x * image_w), int(end_lm.y * image_h))
            cv2.line(annotated_image, start_point, end_point, (0, 255, 255), 2)

        # 관절 포인트 그리기
        for lm in hand_landmarks:
            px = int(lm.x * image_w)
            py = int(lm.y * image_h)
            cv2.circle(annotated_image, (px, py), 4, (255, 80, 80), -1)

        if idx < len(handedness_list):
            handedness = handedness_list[idx]
            x_coordinates = [landmark.x for landmark in hand_landmarks]
            y_coordinates = [landmark.y for landmark in hand_landmarks]
            text_x = int(min(x_coordinates) * image_w)
            text_y = int(min(y_coordinates) * image_h) - MARGIN

            cv2.putText(
                annotated_image,
                f"{handedness[0].category_name}",
                (text_x, text_y),
                cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE,
                HANDEDNESS_TEXT_COLOR,
                FONT_THICKNESS,
                cv2.LINE_AA,
            )
    return annotated_image

In [ ]:
from gcc import Mouse

prev_pos = None
d_pos = None


def predict_hand_gesture(detection_result, results):
    global prev_pos, d_pos

    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    if not hand_landmarks_list or results['right'] == 0b0000000:
        prev_pos = None
        return

    for idx in range(len(hand_landmarks_list)):
        if results['right'] != 0b1111100:
            continue
        hand_landmarks = hand_landmarks_list[idx]
        hand_label = handedness_list[idx][0].category_name

        if hand_label != "Right":
            continue

        hans_side = 1
        joint = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])
        v = joint[[0, 5, 9, 13, 17]] - 0.5

        if prev_pos is None:
            prev_pos = v.mean(axis=0)
            d_pos = np.array([0.0, 0.0, 0.0])

        # d_pos = d_pos * 0.5 * (prev_pos - v.mean(axis=0)) * 0.5
        d_pos = (v.mean(axis=0) - prev_pos)
        print(d_pos)
        prev_pos = v.mean(axis=0)

        Mouse.mouse(0, (d_pos[0] * 254).astype(int), (d_pos[1] * 254).astype(int))
        return

    prev_pos = None

In [ ]:
def predict_gesture(model, scaler, feature):

    x = np.array(feature, dtype=np.float32).reshape(1, -1)

    x = scaler.transform(x)

    pred = model(x, traning = False).numpy()[0]

    return pred


def predict_hand_gesture_by_dnn(detection_result, model, scalar):

    left_result = None
    right_result = None

    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    if not hand_landmarks_list:
        return {"left": None, "right": None}
    

    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]
        hand_label = handedness_list[idx][0].category_name

        hand_side = -1
        if hand_label == "Right":
            hand_side = 1
        elif hand_label == "Left":
            hand_side = 0
        # hand_side = hand_side * 100

        joint = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])

        v1 = joint[[0,1,2,3,0,5,6,7,0,9,10,11,0,13,14,15,0,17,18,19],:]
        v2 = joint[[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],:]

        v = v2 - v1
        v = v / np.linalg.norm(v, axis=1)[:, np.newaxis]

        angle = np.arccos(np.einsum(
            'nt,nt->n',
            v[[0,1,2,4,5,6,8,9,10,12,13,14,16,17,18],:],
            v[[1,2,3,5,6,7,9,10,11,13,14,15,17,18,19],:]
        ))

        angle = np.degrees(angle)

        # TODO It will be used later... maybe?..
        # v_p1 = joint[[0, 0], :]
        # v_p2 = joint[[5, 17], :]
        # v_palm = v_p2 - v_p1

        # palm_normal = np.cross(v_palm[0], v_palm[1])
        # palm_normal = palm_normal / np.linalg.norm(palm_normal)
        # palm_normal = palm_normal * 500
        # full_data = np.append(angle, palm_normal)
        full_data = angle

        palm_size = np.linalg.norm(joint[17] - joint[5]) + 1e-6

        tdi = np.linalg.norm(joint[4] - joint[8]) / palm_size
        tdm = np.linalg.norm(joint[4] - joint[12]) / palm_size

        full_data = np.append(full_data, tdi)
        full_data = np.append(full_data, tdm)
        full_data = np.append(full_data, hand_side)

        data = np.array([full_data], dtype=np.float32)
    
        # print(data)
        # TODO Change model
        # ret, results, neighbours, dist = knn.findNearest(data, 3)
        # print(results)
        # if dist[0][0] < 2000.0:
        pred = predict_gesture(model, scalar, data)
        # label = int(results[0][0])

        if hand_label == "Right":
            right_result = pred
        elif hand_label == "Left":
            left_result = pred

    return {
        "left": left_result,
        "right": right_result
    }


In [ ]:
import tensorflow as tf
import pickle


def load_model(model_file="gesture_model.keras", scalar_file="gesture_scaler.pkl"):
    model = tf.keras.models.load_model(model_file, safe_mode=False, compile=True)
    with open(scalar_file, "rb") as f:
        scaler = pickle.load(f)

    return model, scaler



model, scalar = load_model()

cap = cv2.VideoCapture(CAMERA)
if not cap.isOpened():
    raise RuntimeError(f"웹캠을 열 수 없습니다. camera_index={CAMERA}")

detector = create_hand_detector(model_path="hand_landmarker.task", num_hands=MAX_HANDS)
hand_connections = get_drawing_utils()
# knn = load_knn()
try:
    while True:
        ret, frame = capture_frame(cap)
        if not ret:
            print("프레임을 읽지 못했습니다. 종료합니다.")
            break

        rgb_frame, detection_result = detect_hand_landmarks(detector, frame)
        annotated_image = draw_landmarks_on_image(rgb_frame, detection_result, hand_connections)

        results = predict_hand_gesture_by_dnn(detection_result, model, scalar)
        
        predict_hand_gesture(detection_result, results)
        # TODO Modulization
        handedness_list = getattr(detection_result, "handedness", [])


        # OpenCV는 BGR 기준으로 표시하므로 RGB -> BGR로 변환
        display_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)

        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.imshow("MediaPipe Hand Landmarks", display_image)

        # ESC 또는 q 입력 시 종료
        key = cv2.waitKey(1) & 0xFF
        if key == 27 or key == ord("q"):
            break
        
finally:
    cap.release()
    cv2.waitKey(-1)
    cv2.destroyAllWindows()

Error in cpuinfo: prctl(PR_SVE_GET_VL) failed
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1773643144.805977  352881 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773643144.832740  352881 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773643145.190116  352882 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/rapi07/miniconda3/envs/GCC-rapi/lib/python3.10/site-packages/cv2/qt/plugins"


[0. 0. 0.]
[-0.03124406 -0.16679448  0.00946109]
[-0.02968819 -0.0527613   0.00834515]
[-0.10084983 -0.16743823 -0.00555898]
[-0.05147914 -0.0285607  -0.00202168]
[-0.05627073 -0.01687898  0.00187691]
[-0.10406719 -0.04765083 -0.03013545]
[-0.02619283  0.01281848  0.00969942]
[-0.04122567 -0.00329169  0.00173892]
[-0.03207341 -0.00616127 -0.00238965]
[-0.02937661 -0.00029405  0.00150779]
[-0.01558596  0.00255807 -0.00011142]
[-3.28938961e-03 -3.27914357e-03 -5.43496569e-05]
[ 0.00468243 -0.00683103  0.00101464]
[ 2.87409425e-03 -2.97225118e-03  6.03466939e-05]
[-0.02267724 -0.00891688  0.00506858]
[-0.01574653 -0.0173549   0.00042387]
[-0.02244464 -0.01927692  0.00280598]
[-0.0290123  -0.01745588  0.00549345]
[-0.03747584 -0.01345932  0.00543907]
[-0.02334104 -0.01098078  0.00055144]
[-0.02051975 -0.00719173  0.00133602]
[-0.02595978 -0.00765249  0.00011504]
[-0.01438848 -0.00264326  0.00177073]
[-0.01774127 -0.00312701  0.00303984]
[-1.19827628e-02  6.77704811e-06  2.94707539e-03]
[-0

In [ ]:
detection_result

In [ ]:
import pyautogui
pyautogui.position()